# Persistent Homology with Ripser

**Persistent homology** tracks topological features (components, loops, voids) across a filtration. This notebook:
1. Computes persistent homology with **ripser**
2. Interprets birth-death pairs
3. Compares point clouds with different topology

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from ripser import ripser
    HAS_RIPSER = True
except ImportError:
    HAS_RIPSER = False
    print('ripser not installed -- pip install ripser')

try:
    from persim import plot_diagrams
    HAS_PERSIM = True
except ImportError:
    HAS_PERSIM = False
    print('persim not installed -- pip install persim')

%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 5)

## 1. Generating Point Clouds with Known Topology

- **Circle** ($S^1$): $\beta_0=1, \beta_1=1$
- **Sphere** ($S^2$ sampled in $\mathbb{R}^3$): $\beta_0=1, \beta_1=0, \beta_2=1$
- **Torus** ($T^2$): $\beta_0=1, \beta_1=2, \beta_2=1$

In [ ]:
np.random.seed(0)

# Noisy circle
n = 200
theta = 2 * np.pi * np.random.rand(n)
circle = np.column_stack([np.cos(theta), np.sin(theta)]) + 0.05 * np.random.randn(n, 2)

# Noisy figure-eight (two loops)
theta2 = 2 * np.pi * np.random.rand(n)
half = n // 2
fig8 = np.vstack([
    np.column_stack([np.cos(theta2[:half]) - 1, np.sin(theta2[:half])]),
    np.column_stack([np.cos(theta2[half:]) + 1, np.sin(theta2[half:])])
]) + 0.05 * np.random.randn(n, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(*circle.T, s=5)
axes[0].set_title('Circle (1 loop)')
axes[0].set_aspect('equal')
axes[1].scatter(*fig8.T, s=5)
axes[1].set_title('Figure-Eight (2 loops)')
axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

## 2. Computing Persistent Homology

Ripser computes the Vietoris-Rips filtration and returns **persistence pairs** $(b_i, d_i)$ for each feature.
- **Birth** $b_i$: the scale at which the feature appears
- **Death** $d_i$: the scale at which it merges/fills in
- **Persistence** $d_i - b_i$: how long the feature survives (longer = more significant)

In [ ]:
if HAS_RIPSER:
    # Circle
    result_circle = ripser(circle, maxdim=1)
    dgms_circle = result_circle['dgms']
    
    print("Circle -- H0 pairs:", len(dgms_circle[0]))
    print("Circle -- H1 pairs:", len(dgms_circle[1]))
    
    # Show the most persistent H1 feature
    h1 = dgms_circle[1]
    lifetimes = h1[:, 1] - h1[:, 0]
    idx = np.argmax(lifetimes)
    print(f"Most persistent loop: birth={h1[idx,0]:.3f}, death={h1[idx,1]:.3f}, "
          f"persistence={lifetimes[idx]:.3f}")

In [ ]:
if HAS_RIPSER:
    # Figure-eight
    result_fig8 = ripser(fig8, maxdim=1)
    dgms_fig8 = result_fig8['dgms']
    
    h1_fig8 = dgms_fig8[1]
    lifetimes_fig8 = h1_fig8[:, 1] - h1_fig8[:, 0]
    # Two most persistent loops
    top2 = np.argsort(lifetimes_fig8)[-2:]
    print("Figure-8 -- two most persistent H1 features:")
    for i in top2:
        print(f"  birth={h1_fig8[i,0]:.3f}, death={h1_fig8[i,1]:.3f}, "
              f"persistence={lifetimes_fig8[i]:.3f}")

In [ ]:
if HAS_RIPSER and HAS_PERSIM:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_diagrams(dgms_circle, ax=axes[0], show=False)
    axes[0].set_title('Persistence Diagram -- Circle')
    plot_diagrams(dgms_fig8, ax=axes[1], show=False)
    axes[1].set_title('Persistence Diagram -- Figure-Eight')
    plt.tight_layout()
    plt.show()

## 3. Interpreting the Diagrams

- Points **far from the diagonal** represent significant topological features.
- Points **near the diagonal** are noise.
- The circle has **one** prominent $H_1$ point; the figure-eight has **two**.
- $H_0$ features: all but one die quickly (they merge into a single component).

In [ ]:
if HAS_RIPSER:
    # Barcode representation
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, dgms, title in [(axes[0], dgms_circle, 'Circle'), (axes[1], dgms_fig8, 'Figure-8')]:
        h1_data = dgms[1]
        sorted_idx = np.argsort(h1_data[:, 1] - h1_data[:, 0])[::-1]
        for rank, i in enumerate(sorted_idx[:10]):
            ax.barh(rank, h1_data[i,1] - h1_data[i,0], left=h1_data[i,0], height=0.6)
        ax.set_xlabel('Filtration value')
        ax.set_ylabel('Feature rank')
        ax.set_title(f'H1 Barcode -- {title}')
    plt.tight_layout()
    plt.show()

## Key Takeaways

- **Persistent homology** provides a multiscale summary of topological features.
- **Ripser** is the fastest library for Vietoris-Rips persistent homology.
- Birth-death pairs encode when features appear and disappear.
- Long-lived features are topologically meaningful; short-lived ones are typically noise.

**Next:** Persistence diagrams -- distances, stability, and vectorisation.